# 💸 FINAL PROJECT — **Money Mentor** 💰

> A real personal expense tracker. You just type things like
> *"spent ₹250 on swiggy today"* — the AI logs it, checks your budget,
> and at month-end hands you a full financial health report.
>
> This is a genuinely useful app — the kind people actually install.

## How it works (4 steps)

```
 "spent ₹250 on swiggy"
         │
         ▼
 Step 1: READ THE MESSAGE ── Pydantic ──► Expense(amount, category, payment_mode)
         │
         ▼
 Step 2: LOG & CHECK      ── AGENT + TOOLS ──► log_expense(), check_budget(), total_spent()
         │                                    (simple Python: a list + a dict)
         ▼
 Step 3: EXPERT OPINIONS  ── LCEL parallel ──► Analyst 📊 | Savings Coach 🏦 | Budget Guard 🚨
         │                                      (all 3 analyse AT THE SAME TIME)
         ▼
 Step 4: MONTH-END REPORT ── Pydantic ──► HealthCheck(score, status, biggest_expense, tip)
```

## Concepts used (one line each)

| Concept | What it does here |
|---|---|
| **Pydantic** | Turns messy text like *"₹250 on swiggy"* into typed data |
| **Agentic AI** | The LLM decides when to log expenses and check budgets |
| **LCEL `\|`** | Chains prompts and models together like pipes |
| **RunnableParallel** | Three financial experts analyse your spending simultaneously |

⚠️ Paste your Gemini API key in the setup cell before running.

In [1]:
import os

from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_core.tools import tool
from langchain.agents import create_agent

In [2]:
import os
# Set GEMINI_API_KEY in the environment before running.  
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.2,   
)

## Step 1 — Read the expense message (Pydantic + LCEL)

You type anything: *"ordered pizza ₹450 on credit card last night"*.
The LCEL chain turns it into a clean, typed `Expense` object — no manual data entry.

In [3]:
class Expense(BaseModel):
    '''One expense, extracted from plain English.'''
    amount: float = Field(gt=0, description="Amount spent in rupees")
    category: str = Field(description="One of: food, travel, shopping, entertainment, others")
    payment_mode: str = Field(description="e.g. cash, upi, credit card")


expense_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You extract expense details from casual messages. "
     "Category must be exactly one of: food, travel, shopping, entertainment, others."),
    ("human", "{message}"),
])

expense_chain = expense_prompt | llm.with_structured_output(Expense)

In [4]:
expense = expense_chain.invoke({"message": "Ordered biryani on swiggy for ₹280, paid by UPI"})
print(expense)   # Expense(amount=280.0, category='food', payment_mode='upi')

amount=280.0 category='food' payment_mode='upi'


## Step 2 — Log it & check budget (AGENTIC AI with tools)

### What is a tool?
Just a normal Python function with a docstring. The **agent** reads each docstring
and *decides on its own* which one to call.

Our three tools use **simple, readable logic — no formulas**:
- **`log_expense`** → appends to a list (your digital ledger notebook).
- **`check_budget`** → compares spent vs limit in a dictionary (like checking your wallet).
- **`total_spent`** → adds up the list with `sum()`.

In [5]:
# ─── Your data lives in plain Python objects ───
LEDGER = []   # every logged expense lands here: {'amount': 250, 'category': 'food'}

BUDGETS = {                 # monthly limits per category (edit freely!)
    "food": 3000,
    "travel": 1500,
    "shopping": 2000,
    "entertainment": 1000,
}

@tool
def log_expense(amount: float, category: str) -> str:
    '''Save an expense into the ledger.'''
    LEDGER.append({"amount": amount, "category": category.lower()})
    return f"✅ Logged ₹{amount} under '{category}' ({len(LEDGER)} expenses saved)"

@tool
def check_budget(category: str) -> str:
    '''Check how much budget is left for a spending category this month.'''
    cat = category.lower()
    limit = BUDGETS.get(cat, 1000)                       # unknown categories get ₹1000
    spent = sum(e["amount"] for e in LEDGER if e["category"] == cat)
    left = limit - spent
    if left >= 0:
        return f"{cat}: spent ₹{spent} of ₹{limit} → ₹{left} left ✅"
    return f"{cat}: OVER budget by ₹{-left}! 🚨 Limit was ₹{limit}"

@tool
def total_spent() -> str:
    '''Get the total money spent so far from the ledger.'''
    t = sum(e["amount"] for e in LEDGER)
    return f"Total spent so far: ₹{t} across {len(LEDGER)} expenses"

tools = [log_expense, check_budget, total_spent]

### Test the tools directly first (so you can SEE the simple logic working)

In [6]:
print(log_expense.invoke({"amount": 280, "category": "food"}))
print(log_expense.invoke({"amount": 1200, "category": "shopping"}))
print(check_budget.invoke({"category": "food"}))
print(total_spent.invoke({}))

LEDGER.clear()   # reset — now let the agent do this job properly

✅ Logged ₹280.0 under 'food' (1 expenses saved)
✅ Logged ₹1200.0 under 'shopping' (2 expenses saved)
food: spent ₹280.0 of ₹3000 → ₹2720.0 left ✅
Total spent so far: ₹1480.0 across 2 expenses


In [7]:
money_agent = create_agent(
    llm,
    tools,
    system_prompt=(
        "You are Money Mentor's accountant. "
        "Whenever the user reports an expense you MUST: "
        "1) call log_expense to save it, "
        "2) call check_budget for that category, "
        "3) reply in ONE short line confirming the log and the budget status."
    ),
)

### Use it like a real app — report expenses in plain English

In [8]:
my_week = [
    "Spent ₹350 on zomato food delivery",
    "Auto fare to office, ₹60 cash",
    "Bought shoes online for ₹1800 by credit card",
    "Movie tickets ₹500 via UPI",
]

for msg in my_week:
    reply = money_agent.invoke({"messages": [("user", msg)]})["messages"][-1].content
    print(f"🗣️ {msg}")
    print(f"🤖 {reply}\n")

🗣️ Spent ₹350 on zomato food delivery
🤖 [{'type': 'text', 'text': 'Logged ₹350 under Food; you have ₹2,650 left in your Food budget for this month.', 'extras': {'signature': 'EsQDCsEDAWkUfRMItf+JsEhlsWCWwMh8B+XLNe+TD3VYJfL0Zo1BBSKX4Wp88v4bFcN8gztfaSHJ8jw/LbpsaCiI6vlVPmrv8vbJg5X9WkRhqnSydSg3XdD37uBlbracex141r2IZ11BQJWl2drBmvyal0Cg8C8VloJXlf3DKd7FXGRIUUJYBILa3yFedCeKSMR+xmOQOn08lvxg03u4vMjJb3OCI10kppJ0hrj+dCcEi9PIE1s18Gi1Y+8S4/UgMQgrVIcEONORs34dWA/6P4NPYn9HCZut3YozounoY7FpwcoEWo0UWzaKQu38f6KlB3ge33/KqtSFLmgOridwWVBfRKzcV/TYs/CMALiV6Y0ccfyl5HMxSAxmF5mMJ8eVi24jBaUxWTgxuumGQ6v9RpuPIiAQKFDzNkL0USp4TJAYS689T6mVm7d3IRRTfAF39hvmSznG2JZa46aN2gD3JjoVYkzB9lQ7MGQ5UMsuh1+WlfnwCpKQ/TZN+qN1YQo9TUJyOVA89zLpBBHdTNDAdNI5FPGtf8a5o9nMCj5gsBGE6pejLcSOizfqo+M0uWtPab1INiTjCX73E/Iq4NHf0nA3DdkEReI='}}]

🗣️ Auto fare to office, ₹60 cash
🤖 [{'type': 'text', 'text': 'Logged ₹60 under Transport; you have ₹940 remaining in your Transport budget for this month.', 'extras': {'signature': 'EqACCp0CAWkUfRNDC4mj/+hqYplhT

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash\nPlease retry in 34.814761865s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-Fre

## Step 3 — Expert opinions (LCEL `RunnableParallel`)

Three financial experts review your ledger **at the same time**, thanks to `RunnableParallel`.

In [ ]:
def expert(name: str, role: str):
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"You are {name}. {role} Reply in max 2 short sentences."),
        ("human", "This month's expenses:\n{ledger}"),
    ])
    return prompt | llm | StrOutputParser()

experts_chain = RunnableParallel(
    analyst=expert("Analyst Ravi 📊", "Find which category eats the most money."),
    coach=expert("Savings Coach Meena 🏦", "Give ONE practical tip to save more."),
    guard=expert("Budget Guard Vikram 🚨", "Warn about any overspending danger."),
)

In [ ]:
ledger_text = "\n".join(f"- ₹{e['amount']} on {e['category']}" for e in LEDGER)

opinions = experts_chain.invoke({"ledger": ledger_text})

for name, opinion in opinions.items():
    print(f"{name:>8}: {opinion}\n")

## Step 4 — Month-end health check (Pydantic again)

Everything combines into one clean `HealthCheck` object.
Note the trick: `score` is forced between **0 and 100** — Pydantic won't accept anything else.

In [ ]:
class HealthCheck(BaseModel):
    '''Money Mentor's month-end verdict.'''
    score: int = Field(ge=0, le=100, description="Financial health score out of 100")
    status: str = Field(description="Exactly one of: 'Healthy', 'Watch out', 'Danger'")
    biggest_expense_category: str
    saving_tip: str = Field(description="The single best action to take next month")


report_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are Money Mentor writing the month-end report. "
     "Combine the experts' opinions into an honest verdict. Keep 'saving_tip' under 20 words."),
    ("human",
     "EXPENSES:\n{ledger}\n\nEXPERTS' OPINIONS:\n{opinions}"),
])

report_chain = report_prompt | llm.with_structured_output(HealthCheck)

In [ ]:
report = report_chain.invoke({
    "ledger": ledger_text,
    "opinions": "\n".join(f"{k}: {v}" for k, v in opinions.items()),
})

print("=" * 48)
print("💸  MONEY MENTOR — MONTH-END REPORT".center(48))
print("=" * 48)
for e in LEDGER:
    print(f"   • ₹{e['amount']:>7.0f}  {e['category']}")
print("-" * 48)
bar = "█" * (report.score // 10) + "░" * (10 - report.score // 10)
status_emoji = {"Healthy": "😊", "Watch out": "😐", "Danger": "😱"}.get(report.status, "❓")
print(f"   Health : [{bar}] {report.score}/100 {status_emoji} {report.status}")
print(f"   Biggest: {report.biggest_expense_category}")
print(f"   Tip    : {report.saving_tip}")
print("=" * 48)

## 🎁 Bonus — Try different spending styles

Same pipeline, different habits. Watch the score react.

In [ ]:
spenders = [
    ["₹100 idli breakfast", "₹150 bus pass"],                              # careful spender
    ["₹2500 shopping haul", "₹900 concert ticket", "₹600 food delivery"],  # weekend warrior
    ["₹4000 emergency repair", "₹50 tea"],                                 # unlucky month
]

for style in spenders:
    LEDGER.clear()
    for msg in style:
        e = expense_chain.invoke({"message": msg})
        LEDGER.append({"amount": e.amount, "category": e.category})
    r = report_chain.invoke({
        "ledger": "\n".join(f"- ₹{x['amount']} on {x['category']}" for x in LEDGER),
        "opinions": "(experts unavailable, judge yourself)",
    })
    print(f"{' | '.join(style)}  →  score {r.score}/100 [{r.status}]")